In [1]:
import torch
from torch import nn

In [2]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [3]:
model.load_state_dict(torch.load("encoder_pre_trained_w.pt", weights_only=True))

<All keys matched successfully>

In [ ]:
tensor_x = torch.rand((1, 3, 128, 256), dtype=torch.float32)
torch.onnx.export(
    model,
    (tensor_x,),
    f"{model.__class__.__name__}.onnx",  # filename of the ONNX model
    input_names=["input"],  # Rename inputs for the ONNX model
    dynamo=False,  # True or False to select the exporter to use
)

In [ ]:
!../onnx2c/build/onnx2c vaemodel1.onnx > C/vaemodel1.c